# P03 LLaVA Multimodal Instruction Factory 源码全流程 Notebook

这本 notebook 不是通过 `subprocess` 调壳执行脚本，而是把 `project_3_llava_data` 里的完整流程源码直接展开到 notebook 中。

你可以把它当成一本按项目顺序组织的代码讲义来阅读：

- 主流程脚本按推荐执行顺序排在前面
- 辅助脚本和工具脚本排在后面
- 如果 README 里提到某个脚本但仓库里缺失，也会保留缺口说明

## 项目环境

- 项目目录：`project_3_llava_data`
- 建议 Conda 环境：`p3-llava`

## 本 notebook 收录的源码文件顺序

1. `src/collect_multimodal_assets.py` - 收集多模态资产 [存在]
2. `src/generate_llava_data.py` - 生成指令数据 [存在]
3. `src/alignment.py` - 对齐与补充监督 [存在]
4. `src/interleaved.py` - 构建多图交错样本 [存在]
5. `src/visualize_bbox.py` - 生成 bbox 可视化 [存在]
6. `src/quality_control.py` - 质量控制 [存在]
7. `src/prepare_training_data.py` - 封装训练数据 [存在]
8. `src/evaluate_factory.py` - 评估工厂质量 [存在]
9. `src/run_p3_checks.py` - 项目检查 [存在]
10. `src/pipeline_utils.py` - 辅助脚本 [存在]

## 关键产物

- `data/processed/asset_manifest.jsonl`
- `data/processed/asset_collection_summary.json`
- `data/processed/llava_instruct.jsonl`
- `data/processed/llava_alignment.jsonl`
- `data/processed/llava_interleaved.jsonl`
- `data/processed/quality_audit.jsonl`
- `data/processed/low_quality_flags.jsonl`
- `data/processed/manual_review_samples.jsonl`
- `data/processed/qa_visual_audit.jsonl`
- `data/training/final_llava_dataset.jsonl`
- `data/training/train.jsonl`
- `data/training/val.jsonl`
- `data/training/smoke_test.jsonl`
- `data/training/training_manifest.json`
- `data/reports/p3_report.md`
- `data/reports/p3_metrics.json`
- `data/reports/p3_test_results.json`
- `data/reports/p3_test_report.md`


## 项目 README

下面直接附上项目自带的 `README.md`，方便在 notebook 里连同源码一起看。

# P3 LLaVA Multimodal Instruction Factory

This project builds a small, reproducible multimodal instruction data factory for a LLaVA-style model.

## Scope

The current implementation covers:

1. Project goals and boundaries
   - Image description, region grounding, document-style reading, and multi-image comparison.
   - Small-sample reproducibility with local COCO images and annotations.
2. Image collection and re-description
   - General images from local COCO subsets.
   - Derived document-style cards and chart-style cards for OCR and chart-reading tasks.
3. Instruction construction and alignment
   - Scene description, counting QA, chart QA, OCR summary, and bbox grounding.
   - Multi-image interleaved comparisons.
4. Quality control and spot checks
   - Rule-based validation, low-quality filtering, and bbox visualization outputs.
5. Result presentation and engineering learnings
   - Training split, smoke test, report, metrics, and test artifacts.
6. Extension directions
   - Can be extended to real OCR documents, charts, long-context interleaving, and video QA.

## Environment

Dedicated conda environment:

```bash
conda activate p3-llava
```

Environment files:

- `environment.yml`
- `environment.lock.yml`

Recommended creation commands:

```bash
conda env create -f environment.yml
conda env update -n p3-llava -f environment.lock.yml --prune
```

## Recommended Run Order

```bash
python src/collect_multimodal_assets.py
python src/generate_llava_data.py
python src/alignment.py
python src/interleaved.py
python src/visualize_bbox.py
python src/quality_control.py
python src/prepare_training_data.py
python src/evaluate_factory.py
python src/run_p3_checks.py
```

## Main Outputs

- `data/processed/asset_manifest.jsonl`
- `data/processed/asset_collection_summary.json`
- `data/processed/llava_instruct.jsonl`
- `data/processed/llava_alignment.jsonl`
- `data/processed/llava_interleaved.jsonl`
- `data/processed/quality_audit.jsonl`
- `data/processed/low_quality_flags.jsonl`
- `data/processed/manual_review_samples.jsonl`
- `data/processed/qa_visual_audit.jsonl`
- `data/training/final_llava_dataset.jsonl`
- `data/training/train.jsonl`
- `data/training/val.jsonl`
- `data/training/smoke_test.jsonl`
- `data/training/training_manifest.json`
- `data/reports/p3_report.md`
- `data/reports/p3_metrics.json`
- `data/reports/p3_test_results.json`
- `data/reports/p3_test_report.md`


## 源码目录概览

当前 `src/` 中实际存在的 Python 文件共 `10` 个：

- `src/alignment.py`
- `src/collect_multimodal_assets.py`
- `src/evaluate_factory.py`
- `src/generate_llava_data.py`
- `src/interleaved.py`
- `src/pipeline_utils.py`
- `src/prepare_training_data.py`
- `src/quality_control.py`
- `src/run_p3_checks.py`
- `src/visualize_bbox.py`


## 完整源码展开


### `src/collect_multimodal_assets.py` - 收集多模态资产

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import textwrap
from collections import Counter, defaultdict
from pathlib import Path

from PIL import Image, ImageDraw, ImageFont
from tqdm import tqdm

from pipeline_utils import (
    ANNOTATIONS_DIR,
    CHART_IMAGES_DIR,
    DOCUMENT_IMAGES_DIR,
    IMAGES_DIR,
    PROCESSED_DIR,
    ensure_standard_dirs,
    load_json,
    relative_to_data,
    slugify,
    write_json,
    write_jsonl,
)

OUTPUT_FILE = PROCESSED_DIR / "asset_manifest.jsonl"
SUMMARY_FILE = PROCESSED_DIR / "asset_collection_summary.json"


def wrap_lines(draw: ImageDraw.ImageDraw, text: str, font: ImageFont.ImageFont, width: int) -> list[str]:
    words = text.split()
    lines: list[str] = []
    current = ""
    for word in words:
        candidate = f"{current} {word}".strip()
        text_width = draw.textbbox((0, 0), candidate, font=font)[2]
        if text_width <= width or not current:
            current = candidate
        else:
            lines.append(current)
            current = word
    if current:
        lines.append(current)
    return lines


def render_document_card(output_path: Path, title: str, sections: list[str]) -> None:
    image = Image.new("RGB", (1200, 900), color=(248, 247, 242))
    draw = ImageDraw.Draw(image)
    title_font = ImageFont.load_default()
    body_font = ImageFont.load_default()

    draw.rectangle((36, 36, 1164, 864), outline=(35, 43, 56), width=3)
    draw.text((72, 72), title, fill=(27, 38, 59), font=title_font)

    y = 130
    max_width = 1040
    for section in sections:
        for line in wrap_lines(draw, section, body_font, max_width):
            draw.text((72, y), line, fill=(55, 65, 81), font=body_font)
            y += 26
        y += 12

    image.save(output_path)


def render_chart_card(output_path: Path, title: str, counts: list[tuple[str, int]]) -> None:
    image = Image.new("RGB", (1200, 800), color=(252, 252, 251))
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    draw.text((60, 40), title, fill=(24, 39, 75), font=font)

    if not counts:
        draw.text((60, 120), "No object counts available.", fill=(95, 99, 104), font=font)
        image.save(output_path)
        return

    max_count = max(count for _, count in counts)
    base_x = 240
    y = 120
    for idx, (label, count) in enumerate(counts):
        color = (75 + idx * 18, 130 + idx * 8, 180 - idx * 10)
        bar_width = int(700 * (count / max_count))
        draw.text((60, y + 8), label, fill=(55, 65, 81), font=font)
        draw.rectangle((base_x, y, base_x + bar_width, y + 42), fill=color)
        draw.text((base_x + bar_width + 12, y + 8), str(count), fill=(55, 65, 81), font=font)
        y += 90

    image.save(output_path)


def main() -> None:
    ensure_standard_dirs()

    instances = load_json(ANNOTATIONS_DIR / "instances_val2017.json")
    captions = load_json(ANNOTATIONS_DIR / "captions_val2017.json")

    categories = {item["id"]: item["name"] for item in instances["categories"]}
    images_by_id = {item["id"]: item for item in instances["images"]}

    anns_by_image: dict[int, list[dict]] = defaultdict(list)
    for ann in instances["annotations"]:
        anns_by_image[ann["image_id"]].append(ann)

    captions_by_image: dict[int, list[str]] = defaultdict(list)
    for ann in captions["annotations"]:
        captions_by_image[ann["image_id"]].append(ann["caption"])

    records: list[dict] = []
    image_paths = sorted(IMAGES_DIR.glob("*.jpg"))

    for image_path in tqdm(image_paths, desc="Collecting multimodal assets"):
        image_id = int(image_path.stem)
        image_info = images_by_id[image_id]
        width = image_info["width"]
        height = image_info["height"]
        image_captions = captions_by_image.get(image_id, [])
        primary_caption = image_captions[0] if image_captions else "A natural image from the COCO validation split."

        anns = anns_by_image.get(image_id, [])
        object_counts = Counter(categories[ann["category_id"]] for ann in anns)
        top_objects = object_counts.most_common(5)

        general_record = {
            "asset_id": f"general_{image_path.stem}",
            "asset_type": "general_image",
            "image": relative_to_data(image_path),
            "source_image": image_path.name,
            "image_id": image_id,
            "width": width,
            "height": height,
            "captions": image_captions[:3],
            "primary_caption": primary_caption,
            "object_counts": dict(top_objects),
            "top_annotations": [
                {
                    "label": categories[ann["category_id"]],
                    "bbox_xywh": ann["bbox"],
                    "area": ann["area"],
                }
                for ann in sorted(anns, key=lambda item: item["area"], reverse=True)[:3]
            ],
        }
        records.append(general_record)

        doc_path = DOCUMENT_IMAGES_DIR / f"{image_path.stem}_document.png"
        sections = [
            f"Source image id: {image_id}",
            f"Primary caption: {primary_caption}",
            "Candidate objects: " + ", ".join(f"{label} x{count}" for label, count in top_objects) if top_objects else "Candidate objects: none",
            "Use case: OCR-like reading, caption rewriting, and evidence-based QA.",
        ]
        render_document_card(doc_path, f"P3 Vision Brief {image_path.stem}", sections)
        records.append(
            {
                "asset_id": f"document_{image_path.stem}",
                "asset_type": "document_image",
                "image": relative_to_data(doc_path),
                "source_image": image_path.name,
                "image_id": image_id,
                "title": f"P3 Vision Brief {image_path.stem}",
                "document_sections": sections,
                "primary_caption": primary_caption,
                "object_counts": dict(top_objects),
            }
        )

        chart_path = CHART_IMAGES_DIR / f"{image_path.stem}_chart.png"
        render_chart_card(chart_path, f"Object Count Summary {image_path.stem}", top_objects)
        records.append(
            {
                "asset_id": f"chart_{image_path.stem}",
                "asset_type": "chart_image",
                "image": relative_to_data(chart_path),
                "source_image": image_path.name,
                "image_id": image_id,
                "chart_title": f"Object Count Summary {image_path.stem}",
                "chart_counts": [{"label": label, "count": count} for label, count in top_objects],
                "primary_caption": primary_caption,
            }
        )

    summary = {
        "num_source_images": len(image_paths),
        "num_assets_total": len(records),
        "asset_type_distribution": dict(Counter(record["asset_type"] for record in records)),
    }

    write_jsonl(records, OUTPUT_FILE)
    write_json(summary, SUMMARY_FILE)
    print("✅ 多模态资产收集完成。")
    print(summary)


if __name__ == "__main__":
    main()


### `src/generate_llava_data.py` - 生成指令数据

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from pipeline_utils import PROCESSED_DIR, load_jsonl, write_jsonl

ASSET_FILE = PROCESSED_DIR / "asset_manifest.jsonl"
OUTPUT_FILE = PROCESSED_DIR / "llava_instruct.jsonl"


def describe_objects(object_counts: dict[str, int]) -> str:
    if not object_counts:
        return "The scene does not have reliable object labels, so the answer stays at the scene level."
    parts = [f"{label} x{count}" for label, count in object_counts.items()]
    return "Key objects include " + ", ".join(parts) + "."


def build_general_records(asset: dict) -> list[dict]:
    object_counts = asset.get("object_counts", {})
    top_label, top_count = next(iter(object_counts.items()), ("object", 0))
    description = (
        f"This image shows {asset['primary_caption']} "
        f"{describe_objects(object_counts)}"
    )
    count_answer = (
        f"The most frequent labeled category is {top_label}, with {top_count} annotated instance(s). "
        f"This answer is grounded in the local COCO annotation subset for this image."
    )
    return [
        {
            "id": f"{asset['asset_id']}_describe",
            "image": asset["image"],
            "task_type": "image_description",
            "asset_type": asset["asset_type"],
            "conversations": [
                {"from": "human", "value": "<image>\nPlease describe the scene, the main objects, and their relations."},
                {"from": "gpt", "value": description},
            ],
        },
        {
            "id": f"{asset['asset_id']}_count",
            "image": asset["image"],
            "task_type": "object_count_qa",
            "asset_type": asset["asset_type"],
            "conversations": [
                {"from": "human", "value": "<image>\nWhich object category appears most often, and how many instances are there?"},
                {"from": "gpt", "value": count_answer},
            ],
        },
    ]


def build_document_records(asset: dict) -> list[dict]:
    sections = asset["document_sections"]
    summary = " ".join(sections[:3])
    title = asset["title"]
    object_text = sections[2].replace("Candidate objects: ", "")
    return [
        {
            "id": f"{asset['asset_id']}_ocr_summary",
            "image": asset["image"],
            "task_type": "document_ocr_summary",
            "asset_type": asset["asset_type"],
            "conversations": [
                {"from": "human", "value": "<image>\nRead this document-style image and summarize the key information."},
                {"from": "gpt", "value": f"Title: {title}. Summary: {summary}"},
            ],
        },
        {
            "id": f"{asset['asset_id']}_doc_qa",
            "image": asset["image"],
            "task_type": "document_qa",
            "asset_type": asset["asset_type"],
            "conversations": [
                {"from": "human", "value": "<image>\nWhat title does this page use, and which objects are listed in the brief?"},
                {"from": "gpt", "value": f"The title is {title}. The listed objects are: {object_text}."},
            ],
        },
    ]


def build_chart_records(asset: dict) -> list[dict]:
    counts = asset.get("chart_counts", [])
    if counts:
        top = counts[0]
        second = counts[1] if len(counts) > 1 else None
        top_summary = f"The highest bar is {top['label']} with a count of {top['count']}."
        compare_summary = (
            f"The chart ranks {top['label']} first with {top['count']}, followed by {second['label']} with {second['count']}."
            if second
            else f"The chart only exposes one clear category, {top['label']}."
        )
    else:
        top_summary = "The chart does not contain reliable object bars."
        compare_summary = top_summary

    return [
        {
            "id": f"{asset['asset_id']}_chart_read",
            "image": asset["image"],
            "task_type": "chart_reading",
            "asset_type": asset["asset_type"],
            "conversations": [
                {"from": "human", "value": "<image>\nRead this chart and report the largest category."},
                {"from": "gpt", "value": top_summary},
            ],
        },
        {
            "id": f"{asset['asset_id']}_chart_compare",
            "image": asset["image"],
            "task_type": "chart_comparison",
            "asset_type": asset["asset_type"],
            "conversations": [
                {"from": "human", "value": "<image>\nCompare the top object categories shown in the chart."},
                {"from": "gpt", "value": compare_summary},
            ],
        },
    ]


def main() -> None:
    assets = load_jsonl(ASSET_FILE)
    dataset: list[dict] = []

    for asset in assets:
        if asset["asset_type"] == "general_image":
            dataset.extend(build_general_records(asset))
        elif asset["asset_type"] == "document_image":
            dataset.extend(build_document_records(asset))
        elif asset["asset_type"] == "chart_image":
            dataset.extend(build_chart_records(asset))

    write_jsonl(dataset, OUTPUT_FILE)
    print("✅ LLaVA 基础指令数据生成完成。")
    print({"num_records": len(dataset)})


if __name__ == "__main__":
    main()


### `src/alignment.py` - 对齐与补充监督

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from pipeline_utils import PROCESSED_DIR, load_jsonl, normalize_bbox, write_jsonl

ASSET_FILE = PROCESSED_DIR / "asset_manifest.jsonl"
OUTPUT_FILE = PROCESSED_DIR / "llava_alignment.jsonl"


def main() -> None:
    assets = load_jsonl(ASSET_FILE)
    records: list[dict] = []

    for asset in assets:
        if asset["asset_type"] != "general_image":
            continue

        width = asset["width"]
        height = asset["height"]
        for index, annotation in enumerate(asset.get("top_annotations", []), start=1):
            bbox = normalize_bbox(*annotation["bbox_xywh"], width, height)
            label = annotation["label"]
            records.append(
                {
                    "id": f"{asset['asset_id']}_bbox_{index}",
                    "image": asset["image"],
                    "task_type": "region_grounding",
                    "asset_type": asset["asset_type"],
                    "bbox": bbox,
                    "label": label,
                    "conversations": [
                        {"from": "human", "value": f"<image>\nLocate the {label} in the image and return its bounding box."},
                        {
                            "from": "gpt",
                            "value": f"The {label} is located at [{bbox[0]}, {bbox[1]}, {bbox[2]}, {bbox[3]}].",
                        },
                    ],
                }
            )

    write_jsonl(records, OUTPUT_FILE)
    print("✅ 区域对齐数据生成完成。")
    print({"num_records": len(records)})


if __name__ == "__main__":
    main()


### `src/interleaved.py` - 构建多图交错样本

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from pipeline_utils import PROCESSED_DIR, load_jsonl, write_jsonl

ASSET_FILE = PROCESSED_DIR / "asset_manifest.jsonl"
OUTPUT_FILE = PROCESSED_DIR / "llava_interleaved.jsonl"


def compare_objects(left: dict, right: dict) -> str:
    left_caption = left["primary_caption"]
    right_caption = right["primary_caption"]
    left_objects = ", ".join(f"{label} x{count}" for label, count in left.get("object_counts", {}).items()) or "no strong labels"
    right_objects = ", ".join(f"{label} x{count}" for label, count in right.get("object_counts", {}).items()) or "no strong labels"
    return (
        f"Image 1: {left_caption} Key objects: {left_objects}. "
        f"Image 2: {right_caption} Key objects: {right_objects}. "
        "Both answers are grounded in the local captions and object annotations."
    )


def main() -> None:
    assets = [asset for asset in load_jsonl(ASSET_FILE) if asset["asset_type"] == "general_image"]
    assets = sorted(assets, key=lambda item: item["image"])
    records: list[dict] = []

    for index in range(0, len(assets) - 1, 2):
        left = assets[index]
        right = assets[index + 1]
        records.append(
            {
                "id": f"interleaved_{left['image_id']}_{right['image_id']}",
                "image": [left["image"], right["image"]],
                "task_type": "multi_image_comparison",
                "asset_type": "interleaved_pair",
                "conversations": [
                    {"from": "human", "value": "Image 1: <image>\nImage 2: <image>\nCompare the two scenes and mention shared or distinct objects."},
                    {"from": "gpt", "value": compare_objects(left, right)},
                ],
            }
        )

    write_jsonl(records, OUTPUT_FILE)
    print("✅ 多图交错数据生成完成。")
    print({"num_records": len(records)})


if __name__ == "__main__":
    main()


### `src/visualize_bbox.py` - 生成 bbox 可视化

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path

from PIL import Image, ImageDraw, ImageFont

from pipeline_utils import DATA_DIR, IMAGES_DIR, PROCESSED_DIR, QA_VIZ_DIR, ensure_dir, load_jsonl, write_jsonl

INPUT_FILE = PROCESSED_DIR / "llava_alignment.jsonl"
MANIFEST_FILE = PROCESSED_DIR / "qa_visual_audit.jsonl"


def denormalize_bbox(bbox: list[int], width: int, height: int) -> tuple[int, int, int, int]:
    ymin, xmin, ymax, xmax = bbox
    return (
        int(xmin / 1000 * width),
        int(ymin / 1000 * height),
        int(xmax / 1000 * width),
        int(ymax / 1000 * height),
    )


def main() -> None:
    ensure_dir(QA_VIZ_DIR)
    records = load_jsonl(INPUT_FILE)
    grouped: dict[str, list[dict]] = defaultdict(list)
    for record in records:
        grouped[record["image"]].append(record)

    manifest: list[dict] = []
    font = ImageFont.load_default()

    for image_rel, image_records in grouped.items():
        image_path = DATA_DIR / image_rel
        if not image_path.exists():
            continue

        image = Image.open(image_path).convert("RGB")
        draw = ImageDraw.Draw(image)
        width, height = image.size
        for record in image_records:
            x1, y1, x2, y2 = denormalize_bbox(record["bbox"], width, height)
            draw.rectangle((x1, y1, x2, y2), outline=(255, 64, 64), width=3)
            draw.text((x1 + 4, max(0, y1 - 14)), record["label"], fill=(255, 64, 64), font=font)

        output_path = QA_VIZ_DIR / f"viz_{Path(image_rel).name}"
        image.save(output_path)
        manifest.append(
            {
                "image": image_rel,
                "output_image": output_path.relative_to(DATA_DIR).as_posix(),
                "num_boxes": len(image_records),
                "labels": [record["label"] for record in image_records],
            }
        )

    write_jsonl(manifest, MANIFEST_FILE)
    print("✅ 可视化抽检图生成完成。")
    print({"num_visualizations": len(manifest)})


if __name__ == "__main__":
    main()


### `src/quality_control.py` - 质量控制

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter

from pipeline_utils import DATA_DIR, PROCESSED_DIR, parse_bbox_mentions, load_jsonl, write_json, write_jsonl

INPUT_FILES = {
    "llava_instruct": PROCESSED_DIR / "llava_instruct.jsonl",
    "llava_alignment": PROCESSED_DIR / "llava_alignment.jsonl",
    "llava_interleaved": PROCESSED_DIR / "llava_interleaved.jsonl",
}
AUDIT_FILE = PROCESSED_DIR / "quality_audit.jsonl"
LOW_QUALITY_FILE = PROCESSED_DIR / "low_quality_flags.jsonl"
SUMMARY_FILE = PROCESSED_DIR / "quality_summary.json"
MANUAL_REVIEW_FILE = PROCESSED_DIR / "manual_review_samples.jsonl"


def validate_record(dataset_name: str, record: dict) -> tuple[dict, dict | None]:
    flags: list[str] = []
    image_field = record["image"]
    image_paths = image_field if isinstance(image_field, list) else [image_field]
    missing = [path for path in image_paths if not (DATA_DIR / path).exists()]
    if missing:
        flags.append("missing_image")

    conversations = record.get("conversations", [])
    if len(conversations) < 2:
        flags.append("too_few_turns")
    if conversations and "<image>" not in conversations[0]["value"]:
        flags.append("missing_image_token")

    for turn_index, turn in enumerate(conversations):
        if not turn.get("value", "").strip():
            flags.append(f"empty_turn_{turn_index}")

    if dataset_name == "llava_alignment":
        bbox = record.get("bbox")
        if not bbox or len(bbox) != 4:
            flags.append("missing_bbox")
        else:
            ymin, xmin, ymax, xmax = bbox
            if not (0 <= ymin <= ymax <= 1000 and 0 <= xmin <= xmax <= 1000):
                flags.append("bbox_out_of_range")
        answer_bboxes = parse_bbox_mentions(conversations[-1]["value"])
        if not answer_bboxes:
            flags.append("bbox_not_mentioned_in_answer")

    audit_record = {
        "dataset_name": dataset_name,
        "id": record["id"],
        "task_type": record["task_type"],
        "asset_type": record["asset_type"],
        "passed": not flags,
        "flags": flags,
    }
    low_quality = audit_record if flags else None
    return audit_record, low_quality


def main() -> None:
    audit_records: list[dict] = []
    low_quality_records: list[dict] = []
    manual_review_records: list[dict] = []

    for dataset_name, path in INPUT_FILES.items():
        records = load_jsonl(path)
        for record in records:
            audit_record, low_quality = validate_record(dataset_name, record)
            audit_records.append(audit_record)
            if low_quality:
                low_quality_records.append(low_quality)

        manual_review_records.extend(records[:4])

    summary = {
        "total_records": len(audit_records),
        "passed_records": sum(1 for item in audit_records if item["passed"]),
        "failed_records": len(low_quality_records),
        "dataset_distribution": dict(Counter(item["dataset_name"] for item in audit_records)),
        "task_distribution": dict(Counter(item["task_type"] for item in audit_records)),
    }

    write_jsonl(audit_records, AUDIT_FILE)
    write_jsonl(low_quality_records, LOW_QUALITY_FILE)
    write_jsonl(manual_review_records, MANUAL_REVIEW_FILE)
    write_json(summary, SUMMARY_FILE)
    print("✅ 质量控制与抽检规则执行完成。")
    print(summary)


if __name__ == "__main__":
    main()


### `src/prepare_training_data.py` - 封装训练数据

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict

from pipeline_utils import (
    PROCESSED_DIR,
    TRAINING_DIR,
    deterministic_bucket,
    estimated_tokens,
    ensure_dir,
    load_jsonl,
    write_json,
    write_jsonl,
)

INPUT_FILES = [
    PROCESSED_DIR / "llava_instruct.jsonl",
    PROCESSED_DIR / "llava_alignment.jsonl",
    PROCESSED_DIR / "llava_interleaved.jsonl",
]
LOW_QUALITY_FILE = PROCESSED_DIR / "low_quality_flags.jsonl"
FINAL_FILE = TRAINING_DIR / "final_llava_dataset.jsonl"
TRAIN_FILE = TRAINING_DIR / "train.jsonl"
VAL_FILE = TRAINING_DIR / "val.jsonl"
SMOKE_FILE = TRAINING_DIR / "smoke_test.jsonl"
MANIFEST_FILE = TRAINING_DIR / "training_manifest.json"


def main() -> None:
    ensure_dir(TRAINING_DIR)
    blocked_ids = {record["id"] for record in load_jsonl(LOW_QUALITY_FILE)}

    merged: list[dict] = []
    for path in INPUT_FILES:
        merged.extend(load_jsonl(path))

    final_records = [record for record in merged if record["id"] not in blocked_ids]
    final_records = sorted(final_records, key=lambda item: item["id"])

    train_records: list[dict] = []
    val_records: list[dict] = []
    smoke_by_task: dict[str, list[dict]] = defaultdict(list)

    for record in final_records:
        if deterministic_bucket(record["id"]) < 90:
            train_records.append(record)
        else:
            val_records.append(record)
        if len(smoke_by_task[record["task_type"]]) < 3:
            smoke_by_task[record["task_type"]].append(record)

    smoke_records: list[dict] = []
    for task_type in sorted(smoke_by_task):
        smoke_records.extend(smoke_by_task[task_type])
    smoke_records = smoke_records[:24]

    total_tokens = sum(
        estimated_tokens(" ".join(turn["value"] for turn in record["conversations"]))
        for record in final_records
    )

    manifest = {
        "num_records": len(final_records),
        "num_train_records": len(train_records),
        "num_val_records": len(val_records),
        "num_smoke_records": len(smoke_records),
        "task_distribution": dict(Counter(record["task_type"] for record in final_records)),
        "asset_type_distribution": dict(Counter(record["asset_type"] for record in final_records)),
        "estimated_tokens_total": total_tokens,
    }

    write_jsonl(final_records, FINAL_FILE)
    write_jsonl(train_records, TRAIN_FILE)
    write_jsonl(val_records, VAL_FILE)
    write_jsonl(smoke_records, SMOKE_FILE)
    write_json(manifest, MANIFEST_FILE)
    print("✅ 训练前准备完成。")
    print(manifest)


if __name__ == "__main__":
    main()


### `src/evaluate_factory.py` - 评估工厂质量

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter

from pipeline_utils import PROCESSED_DIR, QA_VIZ_DIR, REPORTS_DIR, TRAINING_DIR, ensure_dir, load_json, load_jsonl, write_json

METRICS_FILE = REPORTS_DIR / "p3_metrics.json"
REPORT_FILE = REPORTS_DIR / "p3_report.md"


def main() -> None:
    ensure_dir(REPORTS_DIR)
    assets = load_jsonl(PROCESSED_DIR / "asset_manifest.jsonl")
    llava_instruct = load_jsonl(PROCESSED_DIR / "llava_instruct.jsonl")
    alignment = load_jsonl(PROCESSED_DIR / "llava_alignment.jsonl")
    interleaved = load_jsonl(PROCESSED_DIR / "llava_interleaved.jsonl")
    quality = load_json(PROCESSED_DIR / "quality_summary.json")
    manifest = load_json(TRAINING_DIR / "training_manifest.json")

    metrics = {
        "num_assets_total": len(assets),
        "asset_type_distribution": dict(Counter(asset["asset_type"] for asset in assets)),
        "num_base_records": len(llava_instruct),
        "num_alignment_records": len(alignment),
        "num_interleaved_records": len(interleaved),
        "quality_pass_rate": round(quality["passed_records"] / max(1, quality["total_records"]), 4),
        "qa_visualizations": len(list(QA_VIZ_DIR.glob("viz_*"))),
        "training_manifest": manifest,
        "estimated_manual_review_hours": round(quality["total_records"] * 0.5 / 60, 2),
        "estimated_manual_review_cost_rmb": round(quality["total_records"] * 0.5 * 120 / 60, 2),
        "estimated_external_caption_cost_usd": round(len(assets) * 0.015, 2),
    }
    write_json(metrics, METRICS_FILE)

    report = f"""# P3 LLaVA Multimodal Instruction Factory Report

## 1. 项目背景与目标

- 构建一个小型、可复现的多模态指令数据工厂，覆盖图像描述、区域定位、文档阅读和多图比较。
- 当前样本边界：基于 29 张本地图像及其 COCO 标注，扩展出通用图像、文档风格图像和图表风格图像三类资产。

## 2. 图像采集与重描述

- 原始通用图像：{metrics["asset_type_distribution"].get("general_image", 0)}
- 派生文档图像：{metrics["asset_type_distribution"].get("document_image", 0)}
- 派生图表图像：{metrics["asset_type_distribution"].get("chart_image", 0)}
- 总资产数：{metrics["num_assets_total"]}

## 3. 指令构造与对齐

- 基础指令样本：{metrics["num_base_records"]}
- 区域对齐样本：{metrics["num_alignment_records"]}
- 多图交错样本：{metrics["num_interleaved_records"]}
- 最终训练样本：{manifest["num_records"]}
- 任务分布：{manifest["task_distribution"]}

## 4. 质量控制与抽检

- 规则校验通过率：{metrics["quality_pass_rate"]}
- 质量审计样本数：{quality["total_records"]}
- 低质量样本数：{quality["failed_records"]}
- BBox 可视化抽检图：{metrics["qa_visualizations"]}

## 5. 结果展示与工程经验

- 训练集划分：train={manifest["num_train_records"]} val={manifest["num_val_records"]} smoke={manifest["num_smoke_records"]}
- 估算总 token 数：{manifest["estimated_tokens_total"]}
- 参考外部 caption 重写成本估算：{metrics["estimated_external_caption_cost_usd"]} USD
- 人工抽检工时估算：{metrics["estimated_manual_review_hours"]} 小时
- 人工抽检成本估算：{metrics["estimated_manual_review_cost_rmb"]} 元

## 6. 扩展方向

- 接入真实 OCR、表格和图表数据源，替换当前的派生文档/图表样本。
- 增加更长上下文的图文交错样本，扩展到视频帧和文档页序列。
- 引入教师模型重写与人工复核闭环，升级为更高质量的多模态 SFT 工厂。
"""

    REPORT_FILE.write_text(report, encoding="utf-8")
    print("✅ P3 报告生成完成。")
    print(report)


if __name__ == "__main__":
    main()


### `src/run_p3_checks.py` - 项目检查

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from collections import Counter
from datetime import UTC, datetime
from pathlib import Path

from pipeline_utils import PROCESSED_DIR, QA_VIZ_DIR, ROOT_DIR, TRAINING_DIR, load_json, load_jsonl

SCRIPTS = sorted((ROOT_DIR / "src").glob("*.py"))
REQUIRED_FILES = [
    PROCESSED_DIR / "asset_manifest.jsonl",
    PROCESSED_DIR / "asset_collection_summary.json",
    PROCESSED_DIR / "llava_instruct.jsonl",
    PROCESSED_DIR / "llava_alignment.jsonl",
    PROCESSED_DIR / "llava_interleaved.jsonl",
    PROCESSED_DIR / "quality_audit.jsonl",
    PROCESSED_DIR / "low_quality_flags.jsonl",
    PROCESSED_DIR / "qa_visual_audit.jsonl",
    TRAINING_DIR / "final_llava_dataset.jsonl",
    TRAINING_DIR / "train.jsonl",
    TRAINING_DIR / "val.jsonl",
    TRAINING_DIR / "smoke_test.jsonl",
    TRAINING_DIR / "training_manifest.json",
    ROOT_DIR / "data" / "reports" / "p3_metrics.json",
    ROOT_DIR / "data" / "reports" / "p3_report.md",
]
RESULTS_FILE = ROOT_DIR / "data" / "reports" / "p3_test_results.json"
REPORT_FILE = ROOT_DIR / "data" / "reports" / "p3_test_report.md"


def run_command(command: list[str]) -> dict:
    completed = subprocess.run(command, capture_output=True, text=True)
    return {
        "command": command,
        "returncode": completed.returncode,
        "passed": completed.returncode == 0,
        "stdout": completed.stdout.strip(),
        "stderr": completed.stderr.strip(),
    }


def main() -> None:
    command_checks = []
    py_compile = run_command([sys.executable, "-m", "py_compile", *[str(path) for path in SCRIPTS]])
    py_compile["name"] = "py_compile"
    command_checks.append(py_compile)

    evaluate = run_command([sys.executable, str(ROOT_DIR / "src" / "evaluate_factory.py")])
    evaluate["name"] = "evaluate_factory"
    command_checks.append(evaluate)

    asset_manifest = load_jsonl(PROCESSED_DIR / "asset_manifest.jsonl")
    alignment = load_jsonl(PROCESSED_DIR / "llava_alignment.jsonl")
    interleaved = load_jsonl(PROCESSED_DIR / "llava_interleaved.jsonl")
    train_records = load_jsonl(TRAINING_DIR / "train.jsonl")
    val_records = load_jsonl(TRAINING_DIR / "val.jsonl")
    smoke_records = load_jsonl(TRAINING_DIR / "smoke_test.jsonl")
    manifest = load_json(TRAINING_DIR / "training_manifest.json")
    low_quality = load_jsonl(PROCESSED_DIR / "low_quality_flags.jsonl")
    qa_visuals = list(QA_VIZ_DIR.glob("viz_*"))

    dataset_checks = [
        {
            "name": "required_files_exist",
            "passed": all(path.exists() for path in REQUIRED_FILES),
            "details": {
                "missing_files": [str(path.relative_to(ROOT_DIR)) for path in REQUIRED_FILES if not path.exists()]
            },
        },
        {
            "name": "asset_types_covered",
            "passed": {"general_image", "document_image", "chart_image"} <= {record["asset_type"] for record in asset_manifest},
            "details": {"asset_type_distribution": dict(Counter(record["asset_type"] for record in asset_manifest))},
        },
        {
            "name": "alignment_has_bboxes",
            "passed": all(len(record.get("bbox", [])) == 4 for record in alignment) and len(alignment) > 0,
            "details": {"alignment_records": len(alignment)},
        },
        {
            "name": "interleaved_is_multi_image",
            "passed": all(isinstance(record["image"], list) and len(record["image"]) == 2 for record in interleaved) and len(interleaved) > 0,
            "details": {"interleaved_records": len(interleaved)},
        },
        {
            "name": "train_val_no_overlap",
            "passed": not ({record["id"] for record in train_records} & {record["id"] for record in val_records}),
            "details": {"overlap": len({record["id"] for record in train_records} & {record["id"] for record in val_records})},
        },
        {
            "name": "smoke_covers_multiple_tasks",
            "passed": len({record["task_type"] for record in smoke_records}) >= 4,
            "details": {"smoke_task_distribution": dict(Counter(record["task_type"] for record in smoke_records))},
        },
        {
            "name": "low_quality_records_removed",
            "passed": len(low_quality) == 0,
            "details": {"low_quality_records": len(low_quality)},
        },
        {
            "name": "qa_visualizations_exist",
            "passed": len(qa_visuals) > 0,
            "details": {"qa_visualizations": len(qa_visuals)},
        },
        {
            "name": "manifest_matches_final_count",
            "passed": manifest["num_train_records"] + manifest["num_val_records"] == manifest["num_records"],
            "details": manifest,
        },
    ]

    results = {
        "timestamp_utc": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "overall_passed": all(item["passed"] for item in command_checks) and all(item["passed"] for item in dataset_checks),
        "total_checks": len(command_checks) + len(dataset_checks),
        "passed_checks": sum(item["passed"] for item in command_checks) + sum(item["passed"] for item in dataset_checks),
        "command_checks": command_checks,
        "dataset_checks": dataset_checks,
    }

    RESULTS_FILE.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")

    lines = [
        "# P3 Test Report",
        "",
        f"- Timestamp: {results['timestamp_utc']}",
        f"- Overall status: {'PASS' if results['overall_passed'] else 'FAIL'}",
        f"- Passed checks: {results['passed_checks']}/{results['total_checks']}",
        "",
        "## Command Checks",
        "",
    ]
    for check in command_checks:
        lines.append(f"- {check['name']}: {'PASS' if check['passed'] else 'FAIL'}")
        lines.append(f"  - Command: `{' '.join(check['command'])}`")
        if check["stdout"]:
            lines.append(f"  - Stdout: `{check['stdout'][:600]}`")
        if check["stderr"]:
            lines.append(f"  - Stderr: `{check['stderr'][:600]}`")

    lines.extend(["", "## Dataset Checks", ""])
    for check in dataset_checks:
        lines.append(f"- {check['name']}: {'PASS' if check['passed'] else 'FAIL'}")
        lines.append(f"  - Details: `{json.dumps(check['details'], ensure_ascii=False)}`")

    REPORT_FILE.write_text("\n".join(lines), encoding="utf-8")
    print(json.dumps(results, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


### `src/pipeline_utils.py` - 辅助脚本

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

ROOT_DIR = Path(__file__).resolve().parent.parent
DATA_DIR = ROOT_DIR / "data"
IMAGES_DIR = DATA_DIR / "images"
ANNOTATIONS_DIR = DATA_DIR / "annotations"
DERIVED_DIR = DATA_DIR / "derived"
DOCUMENT_IMAGES_DIR = DERIVED_DIR / "document_images"
CHART_IMAGES_DIR = DERIVED_DIR / "chart_images"
PROCESSED_DIR = DATA_DIR / "processed"
TRAINING_DIR = DATA_DIR / "training"
REPORTS_DIR = DATA_DIR / "reports"
QA_VIZ_DIR = DATA_DIR / "qa_viz"


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_json(data, path: Path) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_jsonl(path: Path) -> list[dict]:
    records: list[dict] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def write_jsonl(records: list[dict], path: Path) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def deterministic_bucket(key: str, buckets: int = 100) -> int:
    digest = hashlib.md5(key.encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % buckets


def estimated_tokens(text: str) -> int:
    compact = re.sub(r"\s+", " ", text).strip()
    return max(1, len(compact) // 4)


def normalize_bbox(x: float, y: float, w: float, h: float, width: int, height: int) -> list[int]:
    xmin = int((x / width) * 1000)
    ymin = int((y / height) * 1000)
    xmax = int(((x + w) / width) * 1000)
    ymax = int(((y + h) / height) * 1000)
    return [
        max(0, min(1000, ymin)),
        max(0, min(1000, xmin)),
        max(0, min(1000, ymax)),
        max(0, min(1000, xmax)),
    ]


def relative_to_data(path: Path) -> str:
    return path.relative_to(DATA_DIR).as_posix()


def slugify(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_") or "item"


def parse_bbox_mentions(text: str) -> list[list[int]]:
    matches = re.findall(r"\[(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\]", text)
    return [[int(part) for part in match] for match in matches]


def ensure_standard_dirs() -> None:
    for path in [
        DERIVED_DIR,
        DOCUMENT_IMAGES_DIR,
        CHART_IMAGES_DIR,
        PROCESSED_DIR,
        TRAINING_DIR,
        REPORTS_DIR,
        QA_VIZ_DIR,
    ]:
        ensure_dir(path)
